#### Import Dependencies

In [3]:
import sys
import os
# Add parent directory ('notebooks') to the search path
sys.path.append(os.path.abspath(os.path.join('..')))


from pydantic import BaseModel, Field

from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery, Document

from langsmith import traceable, get_current_run_tree

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send, Command

from langchain_core.messages import AIMessage, ToolMessage

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional, Sequence
from IPython.display import Image, display
from operator import add
from openai import OpenAI

import openai

import random
import ast
import inspect
import instructor
import json

from utils.utils import get_tool_descriptions, format_ai_message


from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import openai
import fastembed


In [13]:
from qdrant_client import QdrantClient
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
class State(BaseModel):
    expanded_query:List[str]=[]
    retrieved_context:Annotated[List[str],add]=[]
    question_relevent:bool=False
    initial_query:str=''
    answer:str=''
    query:str=""
    k:int=10



In [5]:
class QueryExpandResponse(BaseModel):
    expanded_query: List[str] = Field(
        ..., 
        max_length=4, 
        description="A list of 3 to 4 expanded search queries. Do not exceed 4 items.")

In [ ]:
class RouterResponse(BaseModel):
    is_relevant: bool = Field(
        description="True if the question is about shopping, products, or product categories. False only if the question is completely unrelated to shopping."
    )
    reason: str = Field(
        description="Brief explanation of why the question is or is not relevant."
    )

@traceable(
    name="router_node",
    run_type="llm",
    metadata={"ls_provider": "openai", "ls_model_name": "gpt-4o-mini"}
)
def router_node(state: State) -> dict:
    prompt_template = """You are a router for a shopping assistant that helps users find products.

Your job is to determine whether a user's question is relevant to shopping or product discovery.

A question IS relevant if it:
- Asks about any product category (e.g., shampoos, headphones, shoes)
- Asks about product features, attributes, or benefits (e.g., "color-treated hair protection")
- Compares products or asks for recommendations
- Asks about pricing, affordability, or value

A question is NOT relevant only if it is completely unrelated to shopping (e.g., "What is the capital of France?", "Tell me a joke").

<Question>
{{ query }}
</Question>

Respond with whether the question is relevant and a brief reason.
"""

    template = Template(prompt_template)
    prompt = template.render(query=state.initial_query)

    client = instructor.from_openai(OpenAI())

    response, _ = client.chat.completions.create_with_completion(
        model="gpt-4o-mini",
        response_model=RouterResponse,
        messages=[{"role": "system", "content": prompt}],
        temperature=0,
    )

    return {"question_relevent": response.is_relevant}


def router_conditional_edges(state: State) -> str:
    if state.question_relevent:
        return "query_expand_node"
    return "end_node"


def end_node(state: State) -> dict:
    return {
        "answer": "I can only help with shopping and product-related questions. Please ask me about a product category or specific item you're looking for."
    }


In [6]:
@traceable(
    name="query_expand_node",
    run_type="llm",
    metadata={"ls_provider":"openai","ls_model_name":"gpt-4o-mini"}
)
def query_expand_node(state:State) -> dict:
    
    prompt_template = """You are part of a shopping assistant that can answer questions about products in stock.

Instructions:
- You will be given a question and you need to expand it into a list of statements that can be used in contextual search to retrieve relevant products.
- The statements should not overlap in context.

<Question>
{{ query }}
</Question>
"""
    
    template = Template(prompt_template)
    
    prompt = template.render(
        query=state.initial_query
    )
    
    client = instructor.from_openai(OpenAI())
    
    # NOTE: "gpt-4.1-mini" in the screenshot has a typo. 
    # Use "gpt-4o-mini" to target the correct OpenAI model.
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4o-mini",
        response_model=QueryExpandResponse,
        messages=[{"role": "system", "content": prompt}],
        temperature=0.5,
    )
    
    return {
        "expanded_query": response.expanded_query
    }


In [7]:
def query_expand_conditional_edges(state: State):

    send_messages = []

    for query in state.expanded_query:
        send_messages.append(
            Send(
                "retrieve_data_node",
                {
                    "query": query,
                    "k": 10
                }
            )
        )

    return send_messages


In [8]:
@traceable(
    name="get_embedding",
    run_type="embedding",
    metadata={
        "ls_model": "openai/text-embedding-3-small",
        "ls_provider": "openai",
        "ls_model_type": "embedding"
        }
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding


@traceable(
    name="retrieve_data_node",
    run_type="retriever",
    metadata={
        "ls_provider": "qdrant",

    }
)
def retrieve_data_node(state: State):  
    query_string = state["query"]
    k = state.get("k", 10)
    query_embedding = get_embedding(query_string)

    results = qdrant_client.query_points(
        collection_name="Products-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query_string,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    formatted_context = ""
    for id, chunk, rating in zip(retrieved_context_ids, retrieved_context, retrieved_context_ratings):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        
    return {
        # Wrap the string in a list so LangGraph can concatenate it properly!
        "retrieved_context": [formatted_context] 
    }


    

In [9]:
class AggregatorResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")

@traceable(
    name="aggregator_node",
    run_type="llm",
    metadata={"ls_provider": "openai", "ls_model_name": "gpt-4o-mini"}
)
def aggregator_node(state: State) -> dict:
    preprocessed_context = "\n".join(state.retrieved_context)

    prompt_template = """You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use the word context and refer to it as the available products.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{{ preprocessed_context }}

Question:
{{ question }}
"""

    template = Template(prompt_template)
    
    prompt = template.render(
        preprocessed_context=preprocessed_context,
        question=state.initial_query
    )
    
    client = instructor.from_openai(OpenAI())
    
    # Note: Using AggregatorResponse instead of QueryExpandResponse
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4o-mini",
        response_model=AggregatorResponse,
        messages=[{"role": "system", "content": prompt}],
        temperature=0.5,
    )
    
    return {
        "answer": response.answer
    }


In [ ]:
workflow = StateGraph(State)

workflow.add_node("router_node", router_node)
workflow.add_node("end_node", end_node)
workflow.add_node("query_expand_node", query_expand_node)
workflow.add_node("retrieve_data_node", retrieve_data_node)
workflow.add_node("aggregator_node", aggregator_node)

workflow.add_edge(START, "router_node")
workflow.add_conditional_edges("router_node", router_conditional_edges, ["query_expand_node", "end_node"])
workflow.add_conditional_edges("query_expand_node", query_expand_conditional_edges, ["retrieve_data_node"])
workflow.add_edge("retrieve_data_node", "aggregator_node")
workflow.add_edge("aggregator_node", END)
workflow.add_edge("end_node", END)

graph = workflow.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [11]:

query = "What are some affordable shampoos that also protect color-treated hair?"

initial_state = {
    "initial_query":query
}

In [14]:
results = graph.invoke(initial_state)

In [15]:
print(results["answer"])

Here are some affordable shampoos that protect color-treated hair:

1. **Pantene Pro-V Color Revival 2in1 Shampoo and Conditioner**  
   - **Rating:** 4.6  
   - **Description:** Helps protect your color-treated hair against dryness and fading. This multitasking formula both cleanses and hydrates hair, making it gentle enough for daily use.  
   - **Features:**  
     - Infuses hair with brilliant shine  
     - Uses Chroma Spectrum technology for color protection  
     - Convenient 2-in-1 formula for easy application  
     - Recommended to use alongside the treatment and shine spray from the Color Revival Collection  

2. **Garnier Fructis Color Shield Instant Color Sealer**  
   - **Rating:** 4.4  
   - **Description:** Proven to stop dry-out and fight fade-out. This lightweight leave-in formula is infused with acai berry and grape seed oil, sealing in color for longer-lasting vibrancy.  
   - **Features:**  
     - Improves manageability and delivers lasting softness  
     - Cont